# Chopp & Cia · 04 — Preparação

**Projeto Integrador VI** · FATEC Votorantim · 2º Semestre/2026

Converte o consolidado em **dataset de modelagem**: define o alvo, aplica a
elegibilidade e separa treino/teste. A saída é enxuta de propósito — só as
colunas que os notebooks 05 a 07 de fato usam, não as 43 do consolidado.

| | |
|:---|:---|
| **Entrada** | consolidado do notebook 01 (CSV) ou a tabela do 02 |
| **Saída** | `dataset_split_v<DATA_VERSION>.csv` (treino+teste) e `dataset_producao_v<DATA_VERSION>.csv` (lookup para escorar quem não treinou) |
| **Ambiente** | Windows local ou Databricks |

### Por que duas saídas, e as duas enxutas

O universo de negócio e o universo de treino não são o mesmo conjunto: o segundo
pode exigir um histórico mínimo (`MIN_COMPRAS`) que o primeiro não exige, porque
um cliente com 1 compra tem taxa de atraso 0% ou 100%, sem meio-termo. Mas esse
mesmo cliente **pode ser escorado** depois de o modelo existir. Por isso:

- `dataset_split` — só quem é elegível, com o alvo, particionado em treino/teste.
- `dataset_producao` — todo o universo de negócio, sem alvo, para aplicar o
  modelo depois de treinado.

> **v1.1 — `MIN_COMPRAS` caiu de 2 para 0.** Não é uma mudança de critério, é a
> mesma régua sobre um dado corrigido. Até a v1.0, `FREQUENCIA_COMPRAS` contava
> movimentação de comodato como compra; com a frequência real a mediana do core
> business caiu de 4 para 1, e o corte antigo deixava 332 clientes com 6 casos na
> classe rara — validação cruzada inviável. O notebook 08 mede o trade-off dessa
> escolha em vez de supô-lo.

Nenhuma das duas carrega coluna que o modelo não usa. Identidade (`ID_PESSOA`)
viaja para auditoria, mas os notebooks 05-07 a excluem antes do fit.

## 1. Painel de preparação

Governa **o que o modelo vê** — não como ele aprende. Mudar `min_compras`, o
alvo ou a lista de features muda o problema; mudar hiperparâmetro nos
notebooks 05-07 muda só aquele modelo.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ── Entrada e saída ──────────────────────────────────────────────────────────
CAMINHO_CSV = r""
PASTA_SAIDA = r""          # vazio: mesma pasta do CSV de entrada
TABELA = "projetointegrador.projetointegrador.dataset_consolidado_v1_0"   # Databricks

DATA_VERSION = "1.0"       # deve bater com a do consolidado; nomeia os arquivos
SOBRESCREVER = False

RANDOM_STATE = 42          # semente única, propagada ao split — reprodutibilidade,
                            # não hiperparâmetro. Trocá-la para "melhorar" a métrica
                            # é escolher o sorteio favorável, não um modelo melhor.

try:
    spark                                                      # noqa: F821
    EM_DATABRICKS = True
except NameError:
    EM_DATABRICKS = False

# ── Universo de modelagem: quais LINHAS entram ───────────────────────────────
MIN_COMPRAS = 0             # exclusivo: mantém quem tem 1+ compra.
                             # A mediana do core é 1 compra e 54% da base tem no
                             # máximo uma. Exigir 3+ (MIN_COMPRAS = 2) deixa 332
                             # clientes com 6 casos na classe rara — 1,2 por fold,
                             # CV de 5 folds inviável. O custo de aceitar todos é
                             # conhecido: quem tem 1 compra só pode ter taxa 0% ou
                             # 100%. O notebook 08 mede esse trade-off em vez de
                             # supô-lo. Ver EDA, seção 12.
SO_CORE_BUSINESS = True      # recorte de negócio: só chopp/chopeira

JANELA_DIAS = None          # recorte TEMPORAL por DIAS_DESDE_ULTIMA_COMPRA.
                             # None = sem corte (comportamento padrão).
                             # Medido no consolidado v1.0, sobre o core com 1+
                             # compra (1.279 clientes, classe rara 129):
                             #    730d → 1.159 clientes, rara 100  (20,0/fold)
                             #    365d →   705 clientes, rara  51  (10,2/fold)
                             #    180d →   336 clientes, rara  26  ( 5,2/fold)
                             #     90d →   202 clientes, rara  16  ( 3,2/fold) ✗
                             # Abaixo de 180 a CV de 5 folds não se sustenta.
                             # Atenção: a prevalência SOBE com o corte (90% sem
                             # janela → 93% em 365d) — cliente recente é mais
                             # arriscado nesta régua, não menos. O eixo população
                             # do notebook 08 varre estes mesmos valores; use-o
                             # para escolher, em vez de fixar no chute.

# ── Alvo: o que o modelo aprende a prever ────────────────────────────────────
# Regra de negócio, não estatística. Mudar o limite ou o combinador redefine o
# problema — métricas de rodadas com definições diferentes não são comparáveis.
LIMITE_ATRASO = 0.20
COMBINADOR = "OU"           # "OU": atraso financeiro OU de comodato já classifica

# ── Features: quais COLUNAS o modelo vê ──────────────────────────────────────
FEATURES_NUM = [
    "FREQUENCIA_COMPRAS", "TICKET_MEDIO", "TOTAL_GASTO",
    "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA",
    "TOTAL_PARCELAS", "TOTAL_COMODATOS",
    "PCT_COMPRAS_A_PRAZO", "PRAZO_MEDIO_COMODATO", "VALOR_MEDIO_COMODATO",
    # As duas abaixo compartilham origem aritmética com o alvo — vazamento
    # PARCIAL, reconhecido e não removido: ver LIMITACOES abaixo.
    "MEDIA_DIAS_ATRASO_PAG", "MEDIA_DIAS_ATRASO_COM",
]
FEATURES_CAT = [
    "PERFIL", "CIDADE", "PAGAMENTO",
    # SEGMENTO fica fora: a EDA mostrou cadastro ruidoso demais para ensinar.
]
ALL_FEATURES = FEATURES_NUM + FEATURES_CAT

# Colunas que NUNCA podem entrar como feature: constroem o alvo, ou identificam
# a pessoa. A guarda adiante falha se qualquer uma aparecer em ALL_FEATURES.
FEATURES_PROIBIDAS = [
    "ID_PESSOA",
    "TAXA_ATRASO_PAGAMENTO", "TAXA_ATRASO_COMODATO",     # constroem o alvo
    "PARCELAS_ATRASADAS", "COMODATOS_ATRASADOS",         # numeradores do alvo
    "RISCO_FINANCEIRO", "RISCO_COMODATO", "PERFIL_RISCO",  # o alvo, outra escala
    "MAX_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_COM",
    "AGING_PAGAMENTO", "AGING_COMODATO",
]
PADROES_NOMINAIS = ("NOME", "NM_", "FANTASIA", "RAZAO", "CPF", "CNPJ", "EMAIL",
                    "TELEFONE", "ENDERECO")

# ── Partição treino/teste ─────────────────────────────────────────────────────
TEST_SIZE = 0.30            # estratificado: a classe positiva é bem majoritária

CSV_SAIDA = {"sep": ";", "encoding": "utf-8-sig", "index": False}

print(f"Ambiente : {'Databricks' if EM_DATABRICKS else 'Local'}")
print(f"Features : {len(ALL_FEATURES)} ({len(FEATURES_NUM)} num + {len(FEATURES_CAT)} cat)")
print(f"Alvo     : atraso > {LIMITE_ATRASO:.0%} ({COMBINADOR}), min_compras > {MIN_COMPRAS}")
print(f"Janela   : {'sem recorte temporal' if JANELA_DIAS is None else str(JANELA_DIAS) + ' dias desde a última compra'}")

## 2. Carga e contrato

Mais rígido que a EDA: além de existir, cada feature declarada no painel precisa
estar sem `NaN`. Um `NaN` numérico só falharia lá na frente, dentro do
`StandardScaler` do notebook 05, com uma mensagem que não aponta para a causa.

In [ ]:
if EM_DATABRICKS:
    dados = spark.table(TABELA).toPandas()                     # noqa: F821
    origem = TABELA
else:
    if not CAMINHO_CSV.strip():
        raise ValueError("Preencha CAMINHO_CSV com o CSV gerado pelo notebook 01.")
    caminho_csv = Path(CAMINHO_CSV.strip())
    if not caminho_csv.exists():
        raise FileNotFoundError(f"CSV não encontrado: {caminho_csv}")
    dados = pd.read_csv(caminho_csv, sep=";", encoding="utf-8-sig", low_memory=False)
    origem = caminho_csv.name

dados.columns = [str(c).strip().upper() for c in dados.columns]

OUTPUT_DIR = (
    Path(PASTA_SAIDA.strip()) if PASTA_SAIDA.strip()
    else (Path(CAMINHO_CSV).parent if not EM_DATABRICKS else Path("."))
)
if not EM_DATABRICKS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUFIXO = DATA_VERSION.replace(".", "_")
CSV_SPLIT = OUTPUT_DIR / f"dataset_split_v{SUFIXO}.csv"
CSV_PRODUCAO = OUTPUT_DIR / f"dataset_producao_v{SUFIXO}.csv"

if not EM_DATABRICKS and CSV_SPLIT.exists() and not SOBRESCREVER:
    raise FileExistsError(
        f"{CSV_SPLIT.name} já existe. Incremente DATA_VERSION ou use SOBRESCREVER = True."
    )

# Vazamento por identificação nominal: checado no painel, não só no dataset —
# uma coluna nominal reintroduzida numa carga futura tem que travar aqui, antes
# de qualquer processamento.
nominais_em_features = sorted(
    c for c in ALL_FEATURES if any(p in c.upper() for p in PADROES_NOMINAIS)
)
if nominais_em_features:
    raise ValueError(
        f"Identificação nominal como feature: {nominais_em_features}. Nome não "
        "prevê inadimplência — o modelo aprenderia clientes específicos, não "
        "comportamento de risco. Remova de FEATURES_NUM/FEATURES_CAT."
    )

vazamento = sorted(set(FEATURES_PROIBIDAS) & set(ALL_FEATURES))
if vazamento:
    raise ValueError(
        f"Vazamento do alvo: {vazamento} constroem ALTO_RISCO e não podem ser "
        "feature — um modelo treinado com elas só recalcula a própria regra."
    )

necessarias = set(ALL_FEATURES) | {
    "ID_PESSOA", "CORE_BUSINESS", "FREQUENCIA_COMPRAS",
    "TAXA_ATRASO_PAGAMENTO", "TAXA_ATRASO_COMODATO",
}
ausentes = sorted(necessarias - set(dados.columns))
if ausentes:
    raise KeyError(
        f"Colunas ausentes em {origem}: {ausentes}. Se você adicionou uma "
        "feature ao painel, acrescente a coluna de origem no notebook 01."
    )

if dados["ID_PESSOA"].duplicated().any():
    raise ValueError("ID_PESSOA duplicado — o consolidado deveria ter 1 linha por cliente.")

for coluna in FEATURES_NUM:
    dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce")
for coluna in FEATURES_CAT:
    dados[coluna] = dados[coluna].fillna("NÃO INFORMADO")

print(f"{len(dados):,} clientes × {dados.shape[1]} colunas em {origem}")

## 3. Universo de modelagem

Filtra o recorte de negócio (core business) e trata ausentes — zero é a leitura
correta para contagem: "nenhuma parcela" são 0 parcelas, não um valor
desconhecido.

In [ ]:
universo = dados.copy()
n0 = len(universo)

if SO_CORE_BUSINESS:
    universo = universo[universo["CORE_BUSINESS"] == 1].copy()
print(f"consolidado          {n0:>6,}")
print(f"core business         {len(universo):>6,}  ({len(universo) - n0:+,})")

# Guardado ANTES da janela: o lookup de produção escora todo o universo de
# negócio, inclusive quem ficou fora do recorte temporal e por isso não treinou.
universo_negocio = universo.copy()

if JANELA_DIAS is not None:
    # Mesma semântica do eixo população do notebook 08: quem nunca comprou tem
    # DIAS_DESDE_ULTIMA_COMPRA nulo e cai fora por definição — o NaN não
    # sobrevive à comparação, e sem compra não há atividade recente.
    n_antes = len(universo)
    universo = universo[universo["DIAS_DESDE_ULTIMA_COMPRA"] <= JANELA_DIAS].copy()
    print(f"janela {JANELA_DIAS}d{'':>10}{len(universo):>6,}  ({len(universo) - n_antes:+,})")

COLS_ZERO = [c for c in FEATURES_NUM if c not in ("TICKET_MEDIO",)]
universo[COLS_ZERO] = universo[COLS_ZERO].fillna(0)
# TICKET_MEDIO fica de fora do zero-fill: cliente sem venda não tem ticket
# médio zero, tem ticket médio inexistente. 0 sugeriria "compra de graça".
universo["TICKET_MEDIO"] = universo["TICKET_MEDIO"].fillna(universo["TICKET_MEDIO"].median())

com_nan = [c for c in FEATURES_NUM if universo[c].isna().any()]
if com_nan:
    raise ValueError(
        f"Features numéricas com NaN após o tratamento: {com_nan}. Verifique a "
        "agregação no notebook 01 — uma coluna que deveria ser 0 está ausente."
    )

print(f"\nuniverso de modelagem {len(universo):>6,} clientes × {len(ALL_FEATURES)} features")

## 4. Alvo e elegibilidade

> **Sobre `MIN_COMPRAS = 0`:** com histórico de 1 compra, a taxa de atraso só
> pode valer 0% ou 100%. O ruído é real e conhecido; a alternativa (cortar) deixa
> a classe rara pequena demais para validar qualquer modelo. O notebook 08 roda os
> dois cenários lado a lado e mostra o custo de cada um em métrica.

`ALTO_RISCO` nasce de `TAXA_ATRASO_PAGAMENTO`/`TAXA_ATRASO_COMODATO` — por isso
essas colunas, e os numeradores que as compõem, não podem ser feature. A
checagem já rodou no painel; aqui falha se o alvo resultar degenerado (todo
mundo na mesma classe), o que travaria a validação cruzada dos notebooks
05-07 com uma mensagem que não aponta para a causa real.

> **Vazamento parcial reconhecido.** `MEDIA_DIAS_ATRASO_PAG/COM` compartilham
> origem aritmética com o alvo. Mantidas conscientemente — as métricas medem a
> capacidade de reproduzir a regra de negócio, não de prever o futuro. Ver
> `LIMITACOES` no fim deste notebook.

In [ ]:
elegiveis = universo[universo["FREQUENCIA_COMPRAS"] > MIN_COMPRAS].copy()
print(f"universo               {len(universo):>6,}")
print(f"elegíveis (>{MIN_COMPRAS} compras)  {len(elegiveis):>6,}  "
      f"({len(elegiveis) - len(universo):+,})")

if len(elegiveis) < 50:
    raise ValueError(
        f"Apenas {len(elegiveis)} clientes elegíveis — amostra pequena demais "
        "para treinar. Reduza MIN_COMPRAS."
    )

cond_pag = elegiveis["TAXA_ATRASO_PAGAMENTO"] > LIMITE_ATRASO
cond_com = elegiveis["TAXA_ATRASO_COMODATO"] > LIMITE_ATRASO
condicao = (cond_pag | cond_com) if COMBINADOR == "OU" else (cond_pag & cond_com)
elegiveis["ALTO_RISCO"] = condicao.astype(int)

n_risco = int(elegiveis["ALTO_RISCO"].sum())
n_bom = len(elegiveis) - n_risco
minoria = min(n_risco, n_bom)

if minoria == 0:
    raise ValueError(
        f"Alvo degenerado: todos os {len(elegiveis)} clientes elegíveis caíram "
        f"na mesma classe. Ajuste LIMITE_ATRASO ou COMBINADOR."
    )

print(f"\nALTO_RISCO=1 (risco)   {n_risco:>6,}  ({n_risco / len(elegiveis) * 100:.0f}%)")
print(f"ALTO_RISCO=0 (bom)     {n_bom:>6,}  ({n_bom / len(elegiveis) * 100:.0f}%)")
print(f"classe minoritária     {minoria:>6,}")

if minoria < 5:
    print(f"\nATENÇÃO: com {minoria} casos na classe rara, uma CV de 5 folds "
          "falha nos notebooks 05-07 (precisa de ao menos 1 caso por fold).")

## 5. Partição treino/teste

Estratificada e com semente fixa. O `n` da classe minoritária no teste governa
a largura de todos os intervalos de confiança dos notebooks seguintes — com
poucos casos, um único cliente reclassificado move a métrica vários pontos.

In [ ]:
X = elegiveis[ALL_FEATURES].copy()
y = elegiveis["ALTO_RISCO"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)

n_pos_teste = int(y_test.sum())
n_neg_teste = int((y_test == 0).sum())
n_min_teste = min(n_pos_teste, n_neg_teste)

print(f"treino  {len(X_train):>5,}  ({(1 - TEST_SIZE) * 100:.0f}%)  "
      f"prevalência {y_train.mean():.1%}")
print(f"teste   {len(X_test):>5,}  ({TEST_SIZE * 100:.0f}%)  "
      f"prevalência {y_test.mean():.1%}")
print(f"\nteste: {n_pos_teste} risco / {n_neg_teste} bom pagador")

if n_min_teste == 0:
    raise ValueError(
        "O teste ficou com 0 casos de uma das classes — nenhuma métrica de "
        "holdout é calculável. Aumente o dataset ou reveja MIN_COMPRAS."
    )
if n_min_teste < 20:
    print(f"\nATENÇÃO: {n_min_teste} casos na classe rara do teste — um cliente "
          f"reclassificado move a métrica ~{100 / n_min_teste:.1f} pontos. "
          "Prefira seleção por CV nos notebooks 05-07, não por holdout.")

## 6. Publicação

Dois arquivos, ambos só com as colunas que o modelo usa — identidade, features
e alvo. Nenhuma das ~43 colunas do consolidado que não entrou no painel viaja
adiante.

In [ ]:
split_saida = pd.concat([
    X_train.assign(ALTO_RISCO=y_train, ID_PESSOA=elegiveis.loc[X_train.index, "ID_PESSOA"],
                   _SPLIT="train"),
    X_test.assign(ALTO_RISCO=y_test, ID_PESSOA=elegiveis.loc[X_test.index, "ID_PESSOA"],
                   _SPLIT="test"),
])
split_saida = split_saida[["ID_PESSOA", *ALL_FEATURES, "ALTO_RISCO", "_SPLIT"]]
split_saida["_DATA_VERSION"] = DATA_VERSION

# Lookup de produção: TODO o universo de negócio, sem alvo e SEM a janela
# temporal — para escorar depois quem tem histórico curto demais ou compra
# antiga demais para ter treinado o modelo, mas ainda pode ser classificado
# por ele. Por isso usa universo_negocio, não universo.
producao_saida = universo_negocio[["ID_PESSOA", *ALL_FEATURES]].copy()
producao_saida["_DATA_VERSION"] = DATA_VERSION

if EM_DATABRICKS:
    spark.createDataFrame(split_saida).write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{TABELA.rsplit('.', 1)[0]}.dataset_split_v{SUFIXO}")   # noqa: F821
    spark.createDataFrame(producao_saida).write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{TABELA.rsplit('.', 1)[0]}.dataset_producao_v{SUFIXO}")  # noqa: F821
    print(f"Tabelas publicadas com sufixo v{SUFIXO}")
else:
    split_saida.to_csv(CSV_SPLIT, **CSV_SAIDA)
    producao_saida.to_csv(CSV_PRODUCAO, **CSV_SAIDA)
    print(f"{CSV_SPLIT.name}     {len(split_saida):>5,} linhas × {split_saida.shape[1]} colunas")
    print(f"{CSV_PRODUCAO.name}  {len(producao_saida):>5,} linhas × {producao_saida.shape[1]} colunas")